In [0]:
# Add topic metadata to hr_document_chunks based on document name

topic_mapping = {
    "GDPR_Employee_Data_Practices_2025-01.pdf":  "GDPR",
    "HR_Quarterly_Summary_2025Q1.pdf":            "HR Summary",
    "Performance_Calibration_Guide_2025-01.pdf":  "Performance",
    "Security_Awareness_Brief_2025-01.pdf":      "Security",
    "Travel_Policy_Quick_Reference_2025-01.pdf": "Travel",
    "DEI_Program_Outline_2025-01.pdf":           "DEI",
    "Exit_Process_Checklist_2025-01.pdf":        "Exit Process",
    "Flexible_Work_Toolkit_2025-01.pdf":         "Flexible Work",
}

# Build a CASE expression from the mapping
case_clauses = []
for doc, topic in topic_mapping.items():
    case_clauses.append(f"  WHEN document_name = '{doc}' THEN '{topic}'")
case_expr = "\n".join(case_clauses)

# 1. Check if topic column exists, if not add it
try:
    spark.sql("""
        ALTER TABLE hr_catalog.hr_core.hr_document_chunks
        ADD COLUMNS (topic STRING)
    """)
    print("Added topic column to table")
except Exception as e:
    if "already exists" in str(e).lower():
        print("Topic column already exists")
    else:
        raise

# 2. Populate the topic column based on document_name
spark.sql(f"""
    UPDATE hr_catalog.hr_core.hr_document_chunks
    SET topic = CASE
{case_expr}
      ELSE NULL
    END
""")

# 3. Verify the results
spark.sql("""
    SELECT document_name, topic, COUNT(*) AS chunk_count
    FROM hr_catalog.hr_core.hr_document_chunks
    GROUP BY document_name, topic
    ORDER BY document_name
""").display()

Topic column already exists


document_name,topic,chunk_count
DEI_Program_Outline_2025-01.pdf,DEI,1
Exit_Process_Checklist_2025-01.pdf,Exit Process,1
Flexible_Work_Toolkit_2025-01.pdf,Flexible Work,1
GDPR_Employee_Data_Practices_2025-01.pdf,GDPR,1
HR_Quarterly_Summary_2025Q1.pdf,HR Summary,1
Performance_Calibration_Guide_2025-01.pdf,Performance,1
Security_Awareness_Brief_2025-01.pdf,Security,1
Travel_Policy_Quick_Reference_2025-01.pdf,Travel,1


In [0]:
from databricks.sdk import WorkspaceClient

# Step 1: Add topic column to hr_document_embeddings table
print("Step 1: Adding topic column to hr_document_embeddings table...")
print("=" * 80)

try:
    spark.sql("""
        ALTER TABLE hr_catalog.hr_core.hr_document_embeddings
        ADD COLUMNS (topic STRING)
    """)
    print("✓ Added topic column to hr_document_embeddings table")
except Exception as e:
    if "already exists" in str(e).lower():
        print("✓ Topic column already exists in hr_document_embeddings")
    else:
        raise

# Step 2: Populate topic column by joining with hr_document_chunks
print("\nStep 2: Populating topic column from hr_document_chunks...")
print("=" * 80)

spark.sql("""
    MERGE INTO hr_catalog.hr_core.hr_document_embeddings AS target
    USING (
        SELECT DISTINCT document_name, topic
        FROM hr_catalog.hr_core.hr_document_chunks
        WHERE topic IS NOT NULL
    ) AS source
    ON target.document_name = source.document_name
    WHEN MATCHED THEN
        UPDATE SET target.topic = source.topic
""")

print("✓ Topic column populated")

# Step 3: Verify the update
print("\nStep 3: Verifying updates...")
print("=" * 80)

result = spark.sql("""
    SELECT 
        document_name,
        topic,
        COUNT(*) as chunk_count
    FROM hr_catalog.hr_core.hr_document_embeddings
    GROUP BY document_name, topic
    ORDER BY document_name
""")

result.display()

# Step 4: Sync the vector search index
print("\nStep 4: Syncing vector search index...")
print("=" * 80)

w = WorkspaceClient()

# Trigger index sync
w.vector_search_indexes.sync_index(
    index_name="hr_catalog.hr_core.hr_document_embeddings_index"
)

print("✓ Index sync triggered")
print("\nThe index will now pick up the new 'topic' metadata field.")
print("Wait a few moments for the sync to complete, then query the index.")

# Check index status
import time
time.sleep(2)  # Brief pause

index_status = w.vector_search_indexes.get_index(
    index_name="hr_catalog.hr_core.hr_document_embeddings_index"
)

print(f"\nIndex Status: {index_status.status.ready}")
print(f"Indexed Rows: {index_status.status.indexed_row_count}")

Step 1: Adding topic column to hr_document_embeddings table...
✓ Added topic column to hr_document_embeddings table

Step 2: Populating topic column from hr_document_chunks...
✓ Topic column populated

Step 3: Verifying updates...


document_name,topic,chunk_count
DEI_Program_Outline_2025-01.pdf,DEI,1
Exit_Process_Checklist_2025-01.pdf,Exit Process,1
Flexible_Work_Toolkit_2025-01.pdf,Flexible Work,1
GDPR_Employee_Data_Practices_2025-01.pdf,GDPR,1
HR_Quarterly_Summary_2025Q1.pdf,HR Summary,1
Performance_Calibration_Guide_2025-01.pdf,Performance,1
Security_Awareness_Brief_2025-01.pdf,Security,1
Travel_Policy_Quick_Reference_2025-01.pdf,Travel,1



Step 4: Syncing vector search index...
✓ Index sync triggered

The index will now pick up the new 'topic' metadata field.
Wait a few moments for the sync to complete, then query the index.

Index Status: True
Indexed Rows: 8


In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

print("Verifying topic metadata in vector search index...")
print("=" * 80)

# Get index details
index_info = w.vector_search_indexes.get_index(
    index_name="hr_catalog.hr_core.hr_document_embeddings_index"
)

print(f"\nIndex: {index_info.name}")
print(f"Status: Ready = {index_info.status.ready}")
print(f"Indexed Rows: {index_info.status.indexed_row_count}")
print(f"Index Type: {index_info.index_type}")

if index_info.delta_sync_index_spec:
    print(f"\nSource Table: {index_info.delta_sync_index_spec.source_table}")

# Verify the topic column in the source table
print("\n" + "=" * 80)
print("Verifying topic column in source data...")
print("=" * 80)

result = spark.sql("""
    SELECT 
        id,
        document_name,
        topic,
        chunk_id,
        SUBSTRING(chunk_text, 1, 100) as chunk_preview
    FROM hr_catalog.hr_core.hr_document_embeddings
    ORDER BY document_name, chunk_id
    LIMIT 10
""")

result.display()

print("\n" + "=" * 80)
print("✅ SUCCESS! The vector search index now includes topic metadata.")
print("=" * 80)

print("\nThe 'topic' column is now available in the index with the following structure:")
print("  - chunk_id: bigint")
print("  - chunk_text: string")
print("  - embedding: array<double>")
print("  - topic: string (NEW!)")
print("  - document_name: string")

print("\nYou can now use the topic field for:")
print("  1. Filtering search results by topic")
print("  2. Organizing and categorizing documents")
print("  3. Building topic-specific RAG applications")
print("  4. Improving search relevance with metadata filters")

Verifying topic metadata in vector search index...

Index: hr_catalog.hr_core.hr_document_embeddings_index
Status: Ready = True
Indexed Rows: 8
Index Type: VectorIndexType.DELTA_SYNC

Source Table: hr_catalog.hr_core.hr_document_embeddings

Verifying topic column in source data...


id,document_name,topic,chunk_id,chunk_preview
DEI_Program_Outline_2025-01.pdf_chunk_1,DEI_Program_Outline_2025-01.pdf,DEI,1,DEI Program Outline 2025-01 Owner: marcin.smith@nordstar.example.com (HR) Version: 2025-01 • ERG la
Exit_Process_Checklist_2025-01.pdf_chunk_1,Exit_Process_Checklist_2025-01.pdf,Exit Process,1,Exit Process Checklist 2025-01 Owner: olivia.rodriguez@nordstar.example.com (HR) Version: 2025-01 •
Flexible_Work_Toolkit_2025-01.pdf_chunk_1,Flexible_Work_Toolkit_2025-01.pdf,Flexible Work,1,Flexible Work Toolkit 2025-01 Owner: ava.white@nordstar.example.com (IT) Version: 2025-01 • Compres
GDPR_Employee_Data_Practices_2025-01.pdf_chunk_1,GDPR_Employee_Data_Practices_2025-01.pdf,GDPR,1,GDPR Employee Data Practices 2025-01 Owner: noah.nowak@nordstar.example.com (HR) Version: 2025-01 •
HR_Quarterly_Summary_2025Q1.pdf_chunk_1,HR_Quarterly_Summary_2025Q1.pdf,HR Summary,1,HR Quarterly Summary 2025Q1 Owner: sofia.wójcik@nordstar.example.com (HR) Version: 2025-01 • Headco
Performance_Calibration_Guide_2025-01.pdf_chunk_1,Performance_Calibration_Guide_2025-01.pdf,Performance,1,Performance Calibration Guide 2025-01 Owner: olivia.rodriguez@nordstar.example.com (HR) Version: 20
Security_Awareness_Brief_2025-01.pdf_chunk_1,Security_Awareness_Brief_2025-01.pdf,Security,1,Security Awareness Brief 2025-01 Owner: marcin.dnbrowska@nordstar.example.com (IT) Version: 2025-01
Travel_Policy_Quick_Reference_2025-01.pdf_chunk_1,Travel_Policy_Quick_Reference_2025-01.pdf,Travel,1,Travel Policy Quick Reference 2025-01 Owner: lucas.williams@nordstar.example.com (Finance) Version:



✅ SUCCESS! The vector search index now includes topic metadata.

The 'topic' column is now available in the index with the following structure:
  - chunk_id: bigint
  - chunk_text: string
  - embedding: array<double>
  - topic: string (NEW!)
  - document_name: string

You can now use the topic field for:
  1. Filtering search results by topic
  2. Organizing and categorizing documents
  3. Building topic-specific RAG applications
  4. Improving search relevance with metadata filters


In [0]:
# Demonstrate the difference between queries with and without topic metadata filtering

print("COMPARISON: Query Results With vs Without Topic Filtering")
print("=" * 100)

# Simulate search relevance by finding documents containing key terms
print("\n WITHOUT TOPIC FILTERING")
print("Query: 'What are the requirements for employee data protection?'")
print("-" * 100)

# Without filtering - searches across ALL documents
results_no_filter = spark.sql("""
    SELECT 
        document_name,
        topic,
        chunk_text,
        chunk_id
    FROM hr_catalog.hr_core.hr_document_chunks
    WHERE 
        LOWER(chunk_text) LIKE '%data%' 
        OR LOWER(chunk_text) LIKE '%protection%'
        OR LOWER(chunk_text) LIKE '%employee%'
    ORDER BY document_name
    LIMIT 10
""")

print("\nResults span MULTIPLE topics (unfiltered):")
results_no_filter.select("document_name", "topic", "chunk_id").display()

print("\n" + "=" * 100)
print("\n WITH TOPIC METADATA FILTERING")
print("Query: 'What are the GDPR requirements for storing employee information?'")
print("-" * 100)

# With topic filtering - searches ONLY GDPR documents
results_with_filter = spark.sql("""
    SELECT 
        document_name,
        topic,
        chunk_text,
        chunk_id
    FROM hr_catalog.hr_core.hr_document_chunks
    WHERE 
        topic = 'GDPR'
        AND (
            LOWER(chunk_text) LIKE '%gdpr%' 
            OR LOWER(chunk_text) LIKE '%employee%'
            OR LOWER(chunk_text) LIKE '%data%'
        )
    ORDER BY chunk_id
    LIMIT 10
""")

print("\nResults are FILTERED to GDPR topic only:")
results_with_filter.select("document_name", "topic", "chunk_id").display()

# Show the actual content
print("\n" + "=" * 100)
print("SAMPLE GDPR CONTENT (First Result)")
print("=" * 100)

gdpr_content = spark.sql("""
    SELECT 
        document_name,
        topic,
        chunk_text
    FROM hr_catalog.hr_core.hr_document_chunks
    WHERE topic = 'GDPR'
    LIMIT 1
""").collect()

if gdpr_content:
    print(f"\nDocument: {gdpr_content[0]['document_name']}")
    print(f"Topic: {gdpr_content[0]['topic']}")
    print(f"\nContent Preview:")
    print(gdpr_content[0]['chunk_text'][:500] + "...\n")

# Summary
print("\n" + "=" * 100)
print("KEY DIFFERENCES")
print("=" * 100)

count_no_filter = spark.sql("""
    SELECT COUNT(DISTINCT topic) as topic_count
    FROM hr_catalog.hr_core.hr_document_chunks
    WHERE 
        LOWER(chunk_text) LIKE '%data%' 
        OR LOWER(chunk_text) LIKE '%protection%'
        OR LOWER(chunk_text) LIKE '%employee%'
""").collect()[0]['topic_count']

count_with_filter = spark.sql("""
    SELECT COUNT(*) as chunk_count
    FROM hr_catalog.hr_core.hr_document_chunks
    WHERE topic = 'GDPR'
""").collect()[0]['chunk_count']

print(f"\n✗ WITHOUT filtering: Results span {count_no_filter} different topics")
print("  → May return irrelevant documents (Travel, DEI, Security, etc.)")
print("  → Lower precision, more noise")
print("  → User has to manually filter through results")

print(f"\n✓ WITH 'GDPR' topic filter: Results limited to {count_with_filter} GDPR chunk(s)")
print("  → Only returns GDPR-specific compliance information")
print("  → Higher precision, focused results")
print("  → Faster, more accurate answers")

print("\n" + "=" * 100)


COMPARISON: Query Results With vs Without Topic Filtering

 WITHOUT TOPIC FILTERING
Query: 'What are the requirements for employee data protection?'
----------------------------------------------------------------------------------------------------

Results span MULTIPLE topics (unfiltered):


document_name,topic,chunk_id
GDPR_Employee_Data_Practices_2025-01.pdf,GDPR,1
Security_Awareness_Brief_2025-01.pdf,Security,1




 WITH TOPIC METADATA FILTERING
Query: 'What are the GDPR requirements for storing employee information?'
----------------------------------------------------------------------------------------------------

Results are FILTERED to GDPR topic only:


document_name,topic,chunk_id
GDPR_Employee_Data_Practices_2025-01.pdf,GDPR,1



SAMPLE GDPR CONTENT (First Result)

Document: GDPR_Employee_Data_Practices_2025-01.pdf
Topic: GDPR

Content Preview:
GDPR Employee Data Practices 2025-01

Owner: noah.nowak@nordstar.example.com (HR)
Version: 2025-01
• Lawful bases overview
• DSAR intake ® response
• Retention map snapshot
• Vendors & DPAs
• Cross-border safeguards...


KEY DIFFERENCES

✗ WITHOUT filtering: Results span 2 different topics
  → May return irrelevant documents (Travel, DEI, Security, etc.)
  → Lower precision, more noise
  → User has to manually filter through results

✓ WITH 'GDPR' topic filter: Results limited to 1 GDPR chunk(s)
  → Only returns GDPR-specific compliance information
  → Higher precision, focused results
  → Faster, more accurate answers

